# LYS v3 — standard 3-D nnU-Net on Kaggle

This notebook runs a LYS-only deployment comparator using the official standard nnU-Net `PlainConvUNet`. It never materializes or evaluates the 57 locked-test cases. The fold-0 five-epoch run is a runtime benchmark only, not scientific model-selection evidence.

Attach `LYS_T2w_manual_v1` (directory or tarball) and the preserved 258-row RatLesNetV2 `split_assignments.csv` or artifact bundle. Enable a Kaggle GPU and Internet.

## 1 — Configuration

Run the five-epoch benchmark first. Set `RUN_FULL_250=True` only after its model size and local CPU inference time are acceptable. `FOLDS_TO_RUN` permits resume across Kaggle sessions.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, Image, display

RUN_SEED = 20260715
BRANCH = "dl-ratlesnetv2-finetune"
REPOSITORY = "https://github.com/paulaize/LYS_PROJ1.git"
NNUNET_COMMIT = "468cf803df9b267150ae2b6c0c59b8ac84f16227"  # v2.8.1
TARGET_ID = 701
TARGET_DATASET = "Dataset701_LYSDevelopmentV1"
NNUNET_PLANNER = "ExperimentPlanner"
NNUNET_PLANS = "nnUNetPlans"
NNUNET_CONFIGURATION = "3d_fullres"
BENCHMARK_TRAINER = "nnUNetTrainer_5epochs"
FULL_TRAINER = "nnUNetTrainer_250epochs"
RUN_BENCHMARK_5E = True
RUN_FULL_250 = False
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
BUILD_RESUME_ARCHIVE = True
SPLIT_ASSIGNMENTS_OVERRIDE = None  # optional absolute Kaggle path
RESUME_ARCHIVE_OVERRIDE = None     # optional absolute Kaggle path

assert set(FOLDS_TO_RUN) <= set(range(5))
assert len(FOLDS_TO_RUN) == len(set(FOLDS_TO_RUN))

WORK = Path("/kaggle/working")
PROJECT = WORK / "LYS_PROJ1"
EXPERIMENT_ROOT = WORK / "LYS_v3_standard3d"
PROVENANCE = EXPERIMENT_ROOT / "provenance"
NNUNET_RAW = WORK / "nnUNet_raw"
NNUNET_PREPROCESSED = WORK / "nnUNet_preprocessed"
NNUNET_RESULTS = WORK / "nnUNet_results"
for key, value in {
    "nnUNet_raw": NNUNET_RAW,
    "nnUNet_preprocessed": NNUNET_PREPROCESSED,
    "nnUNet_results": NNUNET_RESULTS,
}.items():
    os.environ[key] = str(value)

display({
    "run_benchmark_5e": RUN_BENCHMARK_5E,
    "run_full_250": RUN_FULL_250,
    "folds_to_run": FOLDS_TO_RUN,
    "planner": NNUNET_PLANNER,
    "plans": NNUNET_PLANS,
    "configuration": NNUNET_CONFIGURATION,
})

## 2 — Install pinned code and inspect upstream commands

The help calls are intentional: the branch requires inspecting installed external interfaces rather than assuming their arguments.

In [ ]:
if not (PROJECT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
PROJECT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
required = [
    PROJECT / "ratlesnetv2_finetune/scripts/normalize_prepared_dataset.py",
    PROJECT / "ratlesnetv2_finetune/scripts/prepare_architecture_comparator.py",
    PROJECT / "ratlesnetv2_finetune/scripts/calibrate_probability_threshold.py",
    PROJECT / "docs/lys_v3_standard_nnunet_kaggle.md",
    PROJECT / "notebooks/lys_v3_standard_nnunet_kaggle.ipynb",
]
assert all(path.is_file() for path in required), (
    "Push the LYS v3 notebook/protocol commit to the branch before running Kaggle: "
    f"{[str(path) for path in required if not path.is_file()]}"
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/MIC-DKFZ/nnUNet.git@{NNUNET_COMMIT}",
        "nibabel",
        "pandas",
        "scipy",
        "scikit-image",
        "matplotlib",
    ],
    check=True,
)
sys.path.insert(0, str(PROJECT))
assert importlib.metadata.version("nnunetv2") == "2.8.1"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before continuing"
GPU_NAME = torch.cuda.get_device_name(0)
subprocess.run(["nvidia-smi"], check=True)
for command in (
    "nnUNetv2_plan_and_preprocess",
    "nnUNetv2_train",
    "nnUNetv2_predict_from_modelfolder",
):
    assert shutil.which(command), command
    subprocess.run([command, "--help"], check=True, stdout=subprocess.DEVNULL)

print("Project commit:", PROJECT_COMMIT)
print("GPU:", GPU_NAME)
print("nnU-Net:", importlib.metadata.version("nnunetv2"), NNUNET_COMMIT)

## 3 — Restore optional resume state

Attach at most one prior `LYS_v3_standard3d_resume.tar.gz`. It contains checkpoints and reports, never raw images. Raw conversion and preprocessing are rebuilt and verified.

In [ ]:
def safe_extract(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with tarfile.open(archive_path, "r:*") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            assert target == root or root in target.parents, member.name
        archive.extractall(destination)

if RESUME_ARCHIVE_OVERRIDE:
    resume_archives = [Path(RESUME_ARCHIVE_OVERRIDE)]
else:
    resume_archives = sorted(
        Path("/kaggle/input").rglob("LYS_v3_standard3d_resume.tar.gz")
    )
assert len(resume_archives) <= 1, resume_archives
if resume_archives:
    assert not EXPERIMENT_ROOT.exists() and not NNUNET_RESULTS.exists(), (
        "Refusing to merge resume state into existing outputs"
    )
    safe_extract(resume_archives[0], WORK)
    print("Restored:", resume_archives[0])
else:
    print("No resume archive attached.")

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
PROVENANCE.mkdir(exist_ok=True)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

identity = {
    "protocol": "lys_v3_standard3d_deployment_comparator_v1",
    "project_commit": PROJECT_COMMIT,
    "run_seed": RUN_SEED,
    "gpu_name": GPU_NAME,
    "nnunet_version": importlib.metadata.version("nnunetv2"),
    "nnunet_source_commit": NNUNET_COMMIT,
    "planner": NNUNET_PLANNER,
    "plans": NNUNET_PLANS,
    "configuration": NNUNET_CONFIGURATION,
    "target_domain_only": True,
    "locked_test_used": False,
}
identity_path = PROVENANCE / "protocol_identity.json"
if identity_path.is_file():
    assert json.loads(identity_path.read_text()) == identity
else:
    identity_path.write_text(json.dumps(identity, indent=2, sort_keys=True) + "\n")
display(identity)

## 4 — Locate and normalize the LYS prepared dataset

Normalization detects gzip from the payload and validates both mask copies. It writes canonical files only under `/kaggle/working`.

In [ ]:
def locate_or_extract_prepared_dataset(dataset_name: str) -> Path:
    input_root = Path("/kaggle/input")
    matches = sorted(input_root.rglob(f"{dataset_name}/manifest.csv"))
    if len(matches) == 1:
        return matches[0].parent
    assert not matches, f"Multiple exposed {dataset_name} roots: {matches}"
    archives = sorted(input_root.rglob(f"{dataset_name}.tar.gz"))
    assert len(archives) == 1, f"Expected one {dataset_name} archive; found {archives}"
    extract_root = WORK / "uploaded_archives" / dataset_name
    extract_root.mkdir(parents=True, exist_ok=True)
    safe_extract(archives[0], extract_root)
    matches = sorted(extract_root.rglob(f"{dataset_name}/manifest.csv"))
    assert len(matches) == 1, matches
    return matches[0].parent

LYS_NAME = "LYS_T2w_manual_v1"
LYS_SOURCE = locate_or_extract_prepared_dataset(LYS_NAME)
NORMALIZED_BASE = WORK / "normalized_inputs"
subprocess.run(
    [
        sys.executable,
        "-m",
        "ratlesnetv2_finetune.scripts.normalize_prepared_dataset",
        "--input",
        str(LYS_SOURCE),
        "--output-base",
        str(NORMALIZED_BASE),
        "--dataset-name",
        LYS_NAME,
        "--expected-cases",
        "258",
    ],
    cwd=PROJECT,
    check=True,
)
LYS_ROOT = NORMALIZED_BASE / LYS_NAME
assert len(list(LYS_ROOT.rglob("scan.nii.gz"))) == 258
assert (LYS_ROOT / "normalization_report.csv").is_file()
for filename in ("manifest.csv", "normalization_report.csv"):
    portable = PROVENANCE / f"lys_normalized_{filename}"
    if portable.is_file():
        assert sha256(portable) == sha256(LYS_ROOT / filename)
    else:
        shutil.copy2(LYS_ROOT / filename, portable)
print("Normalized LYS root:", LYS_ROOT)

## 5 — Recover and verify the preserved LYS split

The notebook fails if it cannot find the previous 258-row split or if attached copies conflict. It never creates a replacement split.

In [ ]:
def valid_target_split(path: Path):
    try:
        rows = pd.read_csv(path, keep_default_na=False)
    except Exception:
        return None
    needed = {"case_id", "subject_id", "outer_split", "cv_fold"}
    if len(rows) != 258 or not needed <= set(rows.columns) or not rows.case_id.is_unique:
        return None
    if (rows.outer_split == "development").sum() != 201:
        return None
    if (rows.outer_split == "test").sum() != 57:
        return None
    return rows

candidate_paths = []
if SPLIT_ASSIGNMENTS_OVERRIDE:
    candidate_paths.append(Path(SPLIT_ASSIGNMENTS_OVERRIDE))
else:
    candidate_paths.extend(Path("/kaggle/input").rglob("split_assignments.csv"))

split_extract_root = PROVENANCE / "split_candidates"
split_extract_root.mkdir(exist_ok=True)
archive_candidates = sorted(Path("/kaggle/input").rglob("*.tar.gz"))
for archive_index, archive_path in enumerate(archive_candidates):
    if archive_path.name in {"LYS_T2w_manual_v1.tar.gz", "LYS_v3_standard3d_resume.tar.gz"}:
        continue
    try:
        with tarfile.open(archive_path, "r:*") as archive:
            members = [
                member
                for member in archive.getmembers()
                if member.isfile() and member.name.endswith("split_assignments.csv")
            ]
            for member_index, member in enumerate(members):
                handle = archive.extractfile(member)
                assert handle is not None
                output = split_extract_root / f"archive_{archive_index}_{member_index}.csv"
                output.write_bytes(handle.read())
                candidate_paths.append(output)
    except tarfile.TarError:
        continue

accepted = [(path, valid_target_split(path)) for path in candidate_paths if path.is_file()]
accepted = [(path, rows) for path, rows in accepted if rows is not None]
assert accepted, (
    "No preserved 258-row LYS split_assignments.csv found. Attach the prior "
    "RatLesNetV2 artifact or CSV; do not regenerate the split."
)
hash_groups = {}
for path, rows in accepted:
    hash_groups.setdefault(sha256(path), []).append((path, rows))
assert len(hash_groups) == 1, f"Conflicting preserved splits: {list(hash_groups)}"
split_hash, split_copies = next(iter(hash_groups.items()))
SPLIT_ASSIGNMENTS = PROVENANCE / "split_assignments.csv"
if SPLIT_ASSIGNMENTS.is_file():
    assert sha256(SPLIT_ASSIGNMENTS) == split_hash
else:
    shutil.copy2(split_copies[0][0], SPLIT_ASSIGNMENTS)

assignments = pd.read_csv(SPLIT_ASSIGNMENTS, keep_default_na=False)
development = assignments[assignments.outer_split == "development"].copy()
locked = assignments[assignments.outer_split == "test"].copy()
assert assignments.groupby("subject_id").outer_split.nunique().max() == 1
assert development.groupby("subject_id").cv_fold.nunique().max() == 1
assert set(development.cv_fold.astype(str)) == {"0", "1", "2", "3", "4"}
assert set(locked.cv_fold) == {""}
lys_manifest = pd.read_csv(LYS_ROOT / "manifest.csv")
assert set(lys_manifest.case_id) == set(assignments.case_id)
print("Preserved split SHA-256:", split_hash)
display(pd.crosstab([assignments.outer_split, assignments.cv_fold], assignments.cohort))

## 6 — Materialize only the 201 development cases in nnU-Net format

In [ ]:
TARGET_RAW = NNUNET_RAW / TARGET_DATASET
subprocess.run(
    [
        sys.executable,
        "-m",
        "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
        "prepare-target",
        "--input",
        str(LYS_ROOT),
        "--split-assignments",
        str(SPLIT_ASSIGNMENTS),
        "--output",
        str(TARGET_RAW),
        "--dataset-id",
        str(TARGET_ID),
        "--dataset-name",
        "LYSDevelopmentV1",
    ],
    cwd=PROJECT,
    check=True,
)
target_report = json.loads((TARGET_RAW / "conversion_report.json").read_text())
assert target_report["n_training"] == 201
assert target_report["n_locked_test_omitted"] == 57
assert target_report["locked_test_materialized"] is False
assert not (TARGET_RAW / "imagesTs").exists()
for filename in (
    "case_mapping.csv",
    "conversion_report.json",
    "dataset.json",
    "splits_final.json",
):
    portable = PROVENANCE / f"target_{filename}"
    if portable.is_file():
        assert sha256(portable) == sha256(TARGET_RAW / filename)
    else:
        shutil.copy2(TARGET_RAW / filename, portable)
target_mapping = pd.read_csv(TARGET_RAW / "case_mapping.csv", keep_default_na=False)
assert len(target_mapping) == 201
display(target_report)

## 7 — Plan and preprocess the official standard 3-D model

This uses the unmodified default `ExperimentPlanner`. The architecture assertion prevents an accidental ResEnc-M rerun.

In [ ]:
TARGET_PREPROCESSED = NNUNET_PREPROCESSED / TARGET_DATASET
PLAN_PATH = TARGET_PREPROCESSED / f"{NNUNET_PLANS}.json"
PREPROCESS_MARKER = PROVENANCE / "standard3d_preprocessing_complete.json"
previous_plan_hash = (
    json.loads(PREPROCESS_MARKER.read_text())["plans_sha256"]
    if PREPROCESS_MARKER.is_file()
    else None
)
if not PLAN_PATH.is_file():
    subprocess.run(
        [
            "nnUNetv2_plan_and_preprocess",
            "-d",
            str(TARGET_ID),
            "-pl",
            NNUNET_PLANNER,
            "-c",
            NNUNET_CONFIGURATION,
            "-npfp",
            "4",
            "-np",
            "4",
            "--verify_dataset_integrity",
        ],
        check=True,
    )
assert PLAN_PATH.is_file(), PLAN_PATH
if previous_plan_hash is not None:
    assert sha256(PLAN_PATH) == previous_plan_hash, "Rebuilt standard plans changed"
PREPROCESS_MARKER.write_text(
    json.dumps({"plans_sha256": sha256(PLAN_PATH)}, indent=2) + "\n"
)
shutil.copy2(TARGET_RAW / "splits_final.json", TARGET_PREPROCESSED / "splits_final.json")
assert json.loads((TARGET_RAW / "splits_final.json").read_text()) == json.loads(
    (TARGET_PREPROCESSED / "splits_final.json").read_text()
)

plans = json.loads(PLAN_PATH.read_text())
configuration = plans["configurations"][NNUNET_CONFIGURATION]
network_class = configuration["architecture"]["network_class_name"]
assert network_class.endswith("PlainConvUNet"), network_class
assert "ResidualEncoderUNet" not in network_class
plan_summary = {
    "network_class": network_class,
    "plans_sha256": sha256(PLAN_PATH),
    "patch_size": configuration["patch_size"],
    "batch_size": configuration["batch_size"],
    "spacing": configuration["spacing"],
    "median_image_size_in_voxels": configuration["median_image_size_in_voxels"],
    "features_per_stage": configuration["architecture"]["arch_kwargs"]["features_per_stage"],
}
(PROVENANCE / "standard3d_plan_summary.json").write_text(
    json.dumps(plan_summary, indent=2, sort_keys=True) + "\n"
)
display(plan_summary)

## 8 — Resume-safe training helpers

Completed folds are accepted only when `checkpoint_best.pth` and the exact expected validation probability IDs are present.

In [ ]:
RUNS = EXPERIMENT_ROOT / "runs"
RUNS.mkdir(exist_ok=True)

def model_folder(trainer: str, fold: int) -> Path:
    return (
        NNUNET_RESULTS
        / TARGET_DATASET
        / f"{trainer}__{NNUNET_PLANS}__{NNUNET_CONFIGURATION}"
        / f"fold_{fold}"
    )

def expected_validation_ids(fold: int) -> set[str]:
    return set(
        target_mapping.loc[
            target_mapping.cv_fold.astype(str) == str(fold), "nnunet_case_id"
        ]
    )

def train_and_validate(trainer: str, fold: int, marker: Path) -> dict:
    folder = model_folder(trainer, fold)
    best = folder / "checkpoint_best.pth"
    final = folder / "checkpoint_final.pth"
    latest = folder / "checkpoint_latest.pth"
    validation = folder / "validation"
    expected = expected_validation_ids(fold)
    if marker.is_file():
        recorded = json.loads(marker.read_text())
        assert best.is_file()
        assert recorded["checkpoint_best_sha256"] == sha256(best)
        observed = {path.stem for path in validation.glob("*.npz")}
        assert observed == expected
        return recorded

    command = [
        "nnUNetv2_train",
        str(TARGET_ID),
        NNUNET_CONFIGURATION,
        str(fold),
        "-tr",
        trainer,
        "-p",
        NNUNET_PLANS,
    ]
    started = time.time()
    if final.is_file():
        subprocess.run(command + ["--val", "--npz", "--val_best"], check=True)
    else:
        if latest.is_file():
            command.append("--c")
        subprocess.run(command + ["--npz", "--val_best"], check=True)
    elapsed = time.time() - started

    assert best.is_file() and validation.is_dir()
    assert (validation / "summary.json").is_file()
    observed = {path.stem for path in validation.glob("*.npz")}
    assert observed == expected, {
        "missing": sorted(expected - observed),
        "unexpected": sorted(observed - expected),
    }
    checkpoint = torch.load(best, map_location="cpu", weights_only=False)
    parameter_count = sum(
        tensor.numel() for tensor in checkpoint["network_weights"].values()
    )
    record = {
        "trainer": trainer,
        "fold": fold,
        "elapsed_seconds_this_call": elapsed,
        "checkpoint_best": str(best.relative_to(WORK)),
        "checkpoint_best_sha256": sha256(best),
        "checkpoint_best_bytes": best.stat().st_size,
        "parameter_count": parameter_count,
        "n_validation_cases": len(expected),
        "validation_summary": str((validation / "summary.json").relative_to(WORK)),
        "checkpoint_selection": "nnU-Net checkpoint_best; validation exported with --val_best",
        "locked_test_used": False,
        "benchmark_only": trainer == BENCHMARK_TRAINER,
    }
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n")
    return record

## 9 — Fold-0 five-epoch benchmark

This measures training feasibility, parameter count, checkpoint size, and validation execution. It is not accuracy evidence.

In [ ]:
benchmark_record = None
if RUN_BENCHMARK_5E:
    benchmark_record = train_and_validate(
        BENCHMARK_TRAINER,
        0,
        RUNS / "standard3d_5epoch/fold_0/complete.json",
    )
    display(benchmark_record)
    progress = model_folder(BENCHMARK_TRAINER, 0) / "progress.png"
    if progress.is_file():
        display(Image(filename=str(progress)))
else:
    print("Five-epoch benchmark disabled.")

## 10 — Package the five-epoch model for local inference

Download and unzip this archive. Its top-level model folder can be passed directly to `nnUNetv2_predict_from_modelfolder -m ... -f 0 -chk checkpoint_best.pth`.

In [ ]:
BENCHMARK_ZIP = WORK / "LYS_v3_standard3d_5epoch_benchmark.zip"
if RUN_BENCHMARK_5E:
    benchmark_model_root = model_folder(BENCHMARK_TRAINER, 0).parent
    assert (benchmark_model_root / "fold_0/checkpoint_best.pth").is_file()
    with zipfile.ZipFile(
        BENCHMARK_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as archive:
        for path in sorted(benchmark_model_root.rglob("*")):
            if path.is_file() and not path.is_symlink():
                archive.write(path, path.relative_to(benchmark_model_root.parent))
        for path in (
            PROVENANCE / "protocol_identity.json",
            PROVENANCE / "standard3d_plan_summary.json",
            PROVENANCE / "split_assignments.csv",
            RUNS / "standard3d_5epoch/fold_0/complete.json",
            PROJECT / "docs/lys_v3_standard_nnunet_kaggle.md",
            PROJECT / "notebooks/lys_v3_standard_nnunet_kaggle.ipynb",
        ):
            archive.write(path, Path("benchmark_provenance") / path.name)
    print("Benchmark bundle:", BENCHMARK_ZIP, f"{BENCHMARK_ZIP.stat().st_size / 1e6:.1f} MB")
    display(FileLink(str(BENCHMARK_ZIP)))

## 11 — Optional five-fold 250-epoch screen

Run only after the benchmark passes the deployment-size and CPU-runtime gate. Mixing these outputs with canonical 1,000-epoch outputs in one OOF comparison is forbidden.

In [ ]:
full_records = []
if RUN_FULL_250:
    for fold in FOLDS_TO_RUN:
        record = train_and_validate(
            FULL_TRAINER,
            fold,
            RUNS / f"standard3d_250/fold_{fold}/complete.json",
        )
        full_records.append(record)
        display(record)
else:
    print("Full 250-epoch screen disabled.")

## 12 — Export common native-space OOF probabilities when all folds finish

The exports are development validation cases only. Threshold calibration remains OOF-only and postprocessing remains none.

In [ ]:
OOF_ROOT = EXPERIMENT_ROOT / "oof_standard3d_250"
THRESHOLD_ROOT = EXPERIMENT_ROOT / "threshold_standard3d_250"
complete_markers = [
    RUNS / f"standard3d_250/fold_{fold}/complete.json" for fold in range(5)
]
if RUN_FULL_250 and all(path.is_file() for path in complete_markers):
    for fold in range(5):
        validation = model_folder(FULL_TRAINER, fold) / "validation"
        export_root = OOF_ROOT / f"fold_{fold}/prediction_exports/validation/final"
        manifest = export_root / "prediction_export_manifest.csv"
        if not manifest.is_file():
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
                    "export-validation",
                    "--validation",
                    str(validation),
                    "--case-mapping",
                    str(TARGET_RAW / "case_mapping.csv"),
                    "--fold",
                    str(fold),
                    "--output",
                    str(export_root),
                    "--candidate",
                    "nnunetv2_standard3d_250_direct",
                ],
                cwd=PROJECT,
                check=True,
            )
    manifests = sorted(OOF_ROOT.rglob("prediction_export_manifest.csv"))
    rows = pd.concat([pd.read_csv(path) for path in manifests], ignore_index=True)
    assert len(manifests) == 5
    assert len(rows) == 201 and rows.case_id.is_unique
    assert set(rows.case_id) == set(development.case_id)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "ratlesnetv2_finetune.scripts.calibrate_probability_threshold",
            "--prediction-root",
            str(OOF_ROOT),
            "--metadata",
            str(SPLIT_ASSIGNMENTS),
            "--output",
            str(THRESHOLD_ROOT),
            "--thresholds",
            "0.20:0.80:0.05",
            "--surface-tolerance-mm",
            "0.2",
            "--bootstrap-samples",
            "2000",
            "--seed",
            str(RUN_SEED),
            "--overwrite",
        ],
        cwd=PROJECT,
        check=True,
    )
    display(json.loads((THRESHOLD_ROOT / "selected_threshold.json").read_text()))
else:
    print("OOF export waits for all five completed 250-epoch folds.")

## 13 — Build review and resume artifacts

Run this cell at the end of every Kaggle session. The review ZIP is compact and excludes weights. The resume tarball retains the standard-model checkpoints and OOF state but no raw data.

In [ ]:
REVIEW_ZIP = WORK / "LYS_v3_standard3d_review.zip"
allowed_suffixes = {".json", ".csv", ".txt", ".md", ".png", ".pdf", ".ipynb"}
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for root in (EXPERIMENT_ROOT, TARGET_RAW, TARGET_PREPROCESSED):
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if (
                path.is_file()
                and not path.is_symlink()
                and path.suffix.lower() in allowed_suffixes
                and path.stat().st_size <= 25 * 1024 * 1024
            ):
                archive.write(path, path.relative_to(WORK))
    for path in required:
        archive.write(path, Path("repository_snapshot") / path.relative_to(PROJECT))
print("Review bundle:", REVIEW_ZIP, f"{REVIEW_ZIP.stat().st_size / 1e6:.1f} MB")
display(FileLink(str(REVIEW_ZIP)))

if BUILD_RESUME_ARCHIVE:
    RESUME_ARCHIVE = WORK / "LYS_v3_standard3d_resume.tar.gz"
    with tarfile.open(RESUME_ARCHIVE, "w:gz", compresslevel=1) as archive:
        if EXPERIMENT_ROOT.exists():
            archive.add(EXPERIMENT_ROOT, arcname=EXPERIMENT_ROOT.relative_to(WORK))
        dataset_results = NNUNET_RESULTS / TARGET_DATASET
        if dataset_results.exists():
            for trainer in (BENCHMARK_TRAINER, FULL_TRAINER):
                root = dataset_results / f"{trainer}__{NNUNET_PLANS}__{NNUNET_CONFIGURATION}"
                if root.exists():
                    archive.add(root, arcname=root.relative_to(WORK))
    print("Resume bundle:", RESUME_ARCHIVE, f"{RESUME_ARCHIVE.stat().st_size / 1e9:.2f} GB")
    display(FileLink(str(RESUME_ARCHIVE)))

## Interpretation boundary

- The five-epoch output is a runtime benchmark, not evidence of final accuracy.
- Locked-test images were not materialized or evaluated.
- External mouse data were not used.
- Do not mix 250-epoch and canonical 1,000-epoch folds in one OOF comparison.
- Do not enable postprocessing or choose a TTA policy from the locked test.
- Predictions remain draft masks requiring human review.